# PEFT / QLoRA: адаптация LLM к литературным стилям

**Цель:** научить Qwen2.5-1.5B-Instruct различать и воспроизводить стили Чехова и Хармса, сравнить базовую модель и модель с LoRA на отложенных данных. Чехов — реалистическая проза, психологические детали; Хармс — абсурд, парадокс и лаконичность.

Основа: [репозиторий семинара](https://github.com/nebeh/rl_guu/tree/rl_26/seminar/LoRA) и предоставленный Python-скрипт. Сохраняем исходных авторов. В Colab выберите **Среда выполнения → Сменить среду выполнения → T4 GPU** и выполните ячейки по порядку. Нужен интернет для загрузки текстов и модели. Результаты обучения не получены в среде подготовки файла.

In [ ]:
%pip install -q "transformers==4.51.3" "peft==0.15.2" "accelerate==1.6.0" "bitsandbytes==0.45.5"

In [ ]:
import json
import random
import re
import subprocess
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from types import SimpleNamespace
import pandas as pd
import matplotlib.pyplot as plt
REPO_URL = "https://github.com/nebeh/rl_guu.git"
AUTHOR_LABELS = {"chekhov": "Антона Чехова", "kharms": "Даниила Хармса"}
AUTHOR_MARKERS = {
    "chekhov": ("чехов", "chekhov", "chehov", "chekoff", "tchekhov"),
    "kharms": ("хармс", "kharms", "harms", "charms"),
}
SYSTEM = "Ты пишешь оригинальные художественные фрагменты на русском языке."
TOPIC = "Неожиданная встреча на железнодорожной станции"

args = SimpleNamespace(data_dir="rl_guu/seminar/LoRA", model="Qwen/Qwen2.5-1.5B-Instruct", output="lora_result", steps=150, max_examples=160, length=512, eval_examples=8, seed=42, no_clone=False)
set_seed(args.seed)
print("CUDA:", torch.cuda.is_available())

## 1. Подготовка корпуса
Загружаем исходные тексты, определяем автора по имени файла или метаданным, удаляем дубликаты. Разделяем на непересекающиеся фрагменты train/validation. Если доступны только сборники, такое разделение не гарантирует независимость произведений. При наличии нескольких произведений предпочтительно разделять по произведениям.

In [ ]:
def maybe_clone(data_dir: Path, no_clone: bool):
    if data_dir.exists() or no_clone:
        return
    if data_dir.as_posix().replace('\\', '/').endswith("rl_guu/seminar/LoRA"):
        root = data_dir.parent.parent
        print("Каталог данных не найден. Загружаю исходный репозиторий:", REPO_URL)
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "rl_26", REPO_URL, str(root)], check=True)

def author_from_name(value: str):
    name = value.lower().replace("ё", "е")
    for key, markers in AUTHOR_MARKERS.items():
        if any(marker in name for marker in markers):
            return key
    return None

def decode_file(path: Path):
    for enc in ("utf-8-sig", "utf-8", "cp1251"):
        try:
            return path.read_text(encoding=enc)
        except UnicodeError:
            continue
    return ""

def read_documents(root: Path):
    """Тексты ищутся по имени файла/папки; для CSV/JSON возможно поле author."""
    if not root.exists():
        raise FileNotFoundError(f"Каталог с данными не найден: {root}")
    docs = defaultdict(list)
    for path in sorted(root.rglob("*")):
        if not path.is_file() or any(x in path.parts for x in (".git", "__pycache__")):
            continue
        author = author_from_name(str(path.relative_to(root)))
        ext = path.suffix.lower()
        if ext in (".txt", ".md"):
            if author:
                text = decode_file(path)
                if len(text.strip()) >= 200:
                    docs[author].append((path.name, text))
        elif ext in (".jsonl", ".json"):
            try:
                contents = decode_file(path)
                data = [json.loads(line) for line in contents.splitlines() if line.strip()] if ext == ".jsonl" else json.loads(contents)
                if isinstance(data, dict):
                    data = data.get("data", data.get("records", [data]))
                for row in data:
                    if not isinstance(row, dict):
                        continue
                    target_author = author_from_name(str(row.get("author", row.get("style", "")))) or author
                    text = row.get("text", row.get("content", ""))
                    if target_author and isinstance(text, str) and len(text.strip()) >= 200:
                        docs[target_author].append((path.name, text))
            except (ValueError, TypeError, KeyError):
                continue
    return docs

def split_fragments(text: str, minimum: int = 260, maximum: int = 1150):
    """Делит текст по абзацам, без перекрывающихся фрагментов."""
    text = text.replace("\r\n", "\n").replace("\ufeff", "")
    text = re.sub(r"[ \t]+", " ", text)
    paragraphs = [p.strip().replace("\n", " ") for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if not paragraphs:
        return []
    chunks, buffer = [], ""
    for para in paragraphs:
        # Обработка длинных абзацев, которые встречаются в электронных книгах.
        words = para.split()
        pieces, piece = [], ""
        for word in words:
            if len(piece) + len(word) + 1 > maximum and piece:
                pieces.append(piece)
                piece = ""
            piece = (piece + " " + word).strip()
        if piece:
            pieces.append(piece)
        for section in pieces:
            if len(buffer) + len(section) + 2 > maximum and buffer:
                if len(buffer) >= minimum:
                    chunks.append(buffer)
                    buffer = ""
                elif chunks:
                    chunks[-1] = (chunks[-1] + "\n" + buffer).strip()
                    buffer = ""
            buffer = (buffer + "\n" + section).strip()
            if len(buffer) >= maximum * 0.8:
                chunks.append(buffer)
                buffer = ""
    if len(buffer) >= minimum:
        chunks.append(buffer)
    elif buffer and chunks:
        chunks[-1] = (chunks[-1] + "\n" + buffer).strip()
    return [c for c in chunks if len(c) >= minimum]

def make_splits(docs, limit: int, seed: int):
    rng = random.Random(seed)
    train, val = [], []
    for author in AUTHOR_LABELS:
        source = docs.get(author, [])
        if not source:
            raise ValueError(
                f"Нет текстов для {author}. Поместите файлы 'chekhov.txt' и 'kharms.txt' "
                "в --data-dir (или подпапки с такими именами)."
            )
        fragments = []
        for file_name, text in source:
            fragments.extend({"author": author, "text": fragment, "source": file_name}
                             for fragment in split_fragments(text))
        # Убираем полные дубли.
        fragments = list({f["text"]: f for f in fragments}.values())
        rng.shuffle(fragments)
        fragments = fragments[:limit]
        if len(fragments) < 8:
            raise ValueError(f"У {author} только {len(fragments)} фрагментов. Нужно >= 8: добавьте тексты.")
        n_val = max(2, round(len(fragments) * 0.15))
        val.extend(fragments[:n_val])
        train.extend(fragments[n_val:])
        print(f"{author}: документов={len(source)}, train={len(fragments)-n_val}, val={n_val}")
    rng.shuffle(train)
    rng.shuffle(val)
    return train, val

In [ ]:
data_dir = Path(args.data_dir)
maybe_clone(data_dir, args.no_clone)
docs = read_documents(data_dir)
print('Файлы в каталоге:', [str(p.relative_to(data_dir)) for p in data_dir.rglob('*') if p.is_file()][:30])
train, val = make_splits(docs, args.max_examples, args.seed)
assert not ({x['text'] for x in train} & {x['text'] for x in val})
out = Path(args.output)
out.mkdir(exist_ok=True)
(out / 'data_split.json').write_text(json.dumps({'train': train, 'validation': val}, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.crosstab(pd.DataFrame(train+val).author, ['train']*len(train)+['validation']*len(val)))

## 2. Токенизация и маскирование
Промпт задаёт автора. Loss вычисляется только по токенам художественного ответа, а не по инструкции или padding (labels = -100).

In [ ]:
def prompt_text(tokenizer, author: str, topic: str | None = None):
    task = f"Напиши небольшой художественный фрагмент в стиле {AUTHOR_LABELS[author]}."
    if topic:
        task += f" Тема: {topic}."
    task += " Выводи только художественный текст, без пояснений."
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": task}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def encode_example(tokenizer, example, max_length: int, requested_author: str | None = None):
    author = requested_author or example["author"]
    prefix = tokenizer.encode(prompt_text(tokenizer, author), add_special_tokens=False)
    answer = tokenizer.encode(example["text"] + tokenizer.eos_token, add_special_tokens=False)
    if len(prefix) >= max_length - 48:
        raise ValueError("Слишком длинный промпт: увеличьте --length")
    answer = answer[:max_length - len(prefix)]
    ids = prefix + answer
    return {
        "input_ids": ids,
        "attention_mask": [1] * len(ids),
        "labels": [-100] * len(prefix) + answer,
    }

class StyleDataset(Dataset):
    def __init__(self, tokenizer, examples, length):
        self.rows = [encode_example(tokenizer, row, length) for row in examples]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]

class PaddedBatch:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, items):
        n = max(len(x["input_ids"]) for x in items)
        out = {}
        for k in ("input_ids", "attention_mask", "labels"):
            pad = -100 if k == "labels" else (0 if k == "attention_mask" else self.pad_id)
            out[k] = torch.tensor([x[k] + [pad] * (n - len(x[k])) for x in items], dtype=torch.long)
        return out

## 3. PEFT / QLoRA
LoRA добавляет обучаемое низкоранговое обновление ΔW = (α/r)BA к замороженным весам. Используем r=8, α=16, dropout=0.05; на GPU базовая модель квантуется в NF4. Это уменьшает расход памяти.

In [ ]:
def load_model(model_id):
    cuda = torch.cuda.is_available()
    dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if cuda else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    kw = {"torch_dtype": dtype}
    if cuda:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
        )
        kw["device_map"] = {"": torch.cuda.current_device()}
    else:
        print("ВНИМАНИЕ: GPU не обнаружен. CPU-обучение 1.5B может быть очень медленным!")
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    model.config.use_cache = False
    if cuda:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, config)
    model.print_trainable_parameters()
    return model, tokenizer, dtype

if not torch.cuda.is_available():
    raise RuntimeError('Для этого эксперимента включите GPU T4 в Colab.')
model, tokenizer, dtype = load_model(args.model)
ds_train = StyleDataset(tokenizer, train, args.length)
ds_val = StyleDataset(tokenizer, val, args.length)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = model.num_parameters() if not hasattr(model, 'get_nb_trainable_parameters') else model.get_nb_trainable_parameters()[1]
print(f'Обучаемых параметров: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)')

## 4. Оценка базовой модели
Для каждого отложенного фрагмента сравним условную NLL при двух промптах. Стиль с меньшей NLL считается предсказанным. Также генерируем текст на одной новой теме, с одинаковыми параметрами генерации до и после обучения.

In [ ]:
def mean_nll(model, tokenizer, ex, length, target_author):
    row = encode_example(tokenizer, ex, length, requested_author=target_author)
    device = next(model.parameters()).device
    batch = {k: torch.tensor([v], device=device) for k, v in row.items()}
    with torch.inference_mode():
        result = model(**batch)
    return float(result.loss.detach().cpu())

def check_style(model, tokenizer, validation, length, limit_each=8):
    """Held-out conditional NLL and style-choice accuracy for same texts under both prompts."""
    selected = []
    for author in AUTHOR_LABELS:
        selected.extend([x for x in validation if x["author"] == author][:limit_each])
    per_style = defaultdict(list)
    correct = 0
    for ex in selected:
        losses = {author: mean_nll(model, tokenizer, ex, length, author) for author in AUTHOR_LABELS}
        per_style[ex["author"]].append(losses[ex["author"]])
        correct += min(losses, key=losses.get) == ex["author"]
    return {
        "style_accuracy": round(correct / len(selected), 4),
        "conditional_nll": {k: round(sum(v) / len(v), 4) for k, v in per_style.items()},
        "examples_evaluated": len(selected),
    }

def generate_sample(model, tokenizer, author, seed):
    set_seed(seed)
    prompt = prompt_text(tokenizer, author, TOPIC)
    batch = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    device = next(model.parameters()).device
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.inference_mode():
        ids = model.generate(
            **batch, max_new_tokens=160, do_sample=True,
            temperature=0.8, top_p=0.9, repetition_penalty=1.12,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(ids[0][batch["input_ids"].shape[1]:], skip_special_tokens=True).strip()

model.eval()
with model.disable_adapter():
    base_stats = check_style(model, tokenizer, val, args.length, args.eval_examples)
    base_generations = {key: generate_sample(model, tokenizer, key, args.seed + idx) for idx, key in enumerate(AUTHOR_LABELS)}
print(json.dumps(base_stats, ensure_ascii=False, indent=2))

## 5. Обучение адаптера
150 шагов, AdamW (стандарт Trainer), learning rate 2e-4, эффективный batch size 8. Контролируем validation loss каждые 50 шагов.

In [ ]:
model.train()
training_args = TrainingArguments(
    output_dir=str(out / "checkpoints"), max_steps=args.steps,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    weight_decay=0.01, logging_steps=10, eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=1,
    bf16=torch.cuda.is_available() and dtype == torch.bfloat16,
    fp16=torch.cuda.is_available() and dtype == torch.float16,
    report_to="none", remove_unused_columns=False,
    gradient_checkpointing=torch.cuda.is_available(),
    dataloader_num_workers=0,
)
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=PaddedBatch(tokenizer.pad_token_id),
)
print("Запуск обучения...")
trainer.train()
trainer.save_model(str(out / "adapter"))
tokenizer.save_pretrained(str(out / "adapter"))


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(9,4))
for column in ['loss', 'eval_loss']:
    if column in history:
        rows = history.dropna(subset=[column])
        ax.plot(rows['step'], rows[column], marker='o', label=column)
ax.set(xlabel='Шаг', ylabel='Loss', title='Динамика обучения LoRA')
ax.legend(); ax.grid(alpha=.3); plt.show()

## 6. Сравнение до и после
NLL и perplexity должны оцениваться на одинаковых данных. Accuracy характеризует различимость стилевых условий, но не литературное качество. Улучшение не гарантировано.

In [ ]:
model.eval()
model.config.use_cache = True
lora_stats = check_style(model, tokenizer, val, args.length, args.eval_examples)
lora_generations = {key: generate_sample(model, tokenizer, key, args.seed + idx) for idx, key in enumerate(AUTHOR_LABELS)}
metrics = {'base': base_stats, 'lora': lora_stats}
generations = {'base': base_generations, 'lora': lora_generations}
rows = [{'Метрика': 'Style accuracy', 'До': base_stats['style_accuracy'], 'После': lora_stats['style_accuracy']}]
import math
for author in AUTHOR_LABELS:
    a, b = [metrics[k]['conditional_nll'][author] for k in ('base','lora')]
    rows.extend([{'Метрика': 'NLL '+author, 'До': a, 'После': b}, {'Метрика': 'exp(mean NLL) '+author, 'До': math.exp(a), 'После': math.exp(b)}])
comparison = pd.DataFrame(rows).set_index('Метрика')
display(comparison)
comparison.iloc[:1].plot.bar(rot=0, ylim=(0,1), title='Точность различения стиля'); plt.show()
comparison.loc[[i for i in comparison.index if i.startswith('NLL')]].plot.bar(rot=0, title='Условная NLL'); plt.show()
for key, label in AUTHOR_LABELS.items():
    print('\nСтиль:', label, '\nТема:', TOPIC)
    for mode in ('base','lora'):
        print(mode, ':', generations[mode][key], '\n')

## 7. Выводы и сохранение
Автоматический вывод основан на фактических метриках. Дополнительно следует вручную оценить соответствие стилю, связность, оригинальность и отсутствие копирования корпуса.

In [ ]:
def write_report(out: Path, metrics: dict, generations: dict, train: list, val: list, model_name: str, steps: int):
    lines = [
        "# Отчёт: PEFT/LoRA, стили Чехова и Хармса", "",
        f"**Базовая модель:** `{model_name}`", "",
        f"**Метод:** QLoRA (4-bit NF4 на NVIDIA GPU), r=8, alpha=16; базовые веса заморожены.", "",
        f"**Данные:** {len(train)} обучающих фрагментов, {len(val)} отложенных, шагов оптимизации: {steps}.", "",
        "## Количественное сравнение", "",
        "| Метрика | Базовая модель | После LoRA |", "|---|---:|---:|",
        f"| Точность определения стиля по условной NLL | {metrics['base']['style_accuracy']:.2%} | {metrics['lora']['style_accuracy']:.2%} |",
    ]
    for author in AUTHOR_LABELS:
        a = metrics["base"]["conditional_nll"][author]
        b = metrics["lora"]["conditional_nll"][author]
        lines.append(f"| Средняя NLL правильного стиля ({author}) ↓ | {a:.3f} | {b:.3f} |")
    lines.extend([
        "", f"Проверено примеров: {metrics['lora']['examples_evaluated']}.",
        "Оценка проводится по отложенным фрагментам. Для каждого текста вычисляется NLL при двух стилевых промптах; "
        "выбирается стиль с меньшей ошибкой. Это приближённая проверка стилевой обусловленности, "
        "а не доказательство литературного качества.", "",
        "## Качественная проверка на одинаковой теме", "",
        f"Тема: **{TOPIC}**", "",
    ])
    for author in AUTHOR_LABELS:
        lines.extend([
            f"### {AUTHOR_LABELS[author]}", "", "**До LoRA:**", "", generations["base"][author], "",
            "**После LoRA:**", "", generations["lora"][author], "",
        ])
    delta = metrics["lora"]["style_accuracy"] - metrics["base"]["style_accuracy"]
    if delta > 0:
        conclusion = "Точность различения стилей выросла."
    elif delta < 0:
        conclusion = "Точность различения стилей снизилась: стоит проверить качество данных и гиперпараметры."
    else:
        conclusion = "Точность различения стилей не изменилась; необходима дополнительная оценка."
    lines.extend([
        "## Вывод", "", conclusion,
        "Изменение NLL и примеры генерации следует рассматривать совместно.", "",
        "**Ограничение:** если тексты каждого автора представлены одной большой книгой/файлом, "
        "разделение по фрагментам не исключает тематических пересечений train/validation. "
        "Для строгой проверки лучше делить по независимым произведениям и привлекать внешних оценщиков.", "",
    ])
    (out / "report.md").write_text("\n".join(lines), encoding="utf-8")

(out/'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
(out/'generations.json').write_text(json.dumps(generations, ensure_ascii=False, indent=2), encoding='utf-8')
write_report(out, metrics, generations, train, val, args.model, args.steps)
from IPython.display import Markdown, display
display(Markdown((out/'report.md').read_text(encoding='utf-8')))
import shutil
shutil.make_archive('lora_results', 'zip', out)
print('Архив результатов: lora_results.zip')

## 8. Проверка подготовки данных (не требует модели)
Ниже сохранён результат локального теста функций на искусственных строках. Это не результат обучения.

In [1]:
sample = ('Первый тестовый абзац содержит несколько слов. '*30+'\n\n')*5
chunks = split_fragments(sample)
assert chunks and all(len(x)>=260 for x in chunks)
assert author_from_name('chekhov.txt') == 'chekhov'
assert author_from_name('Хармс.txt') == 'kharms'
print('Проверки распознавания авторов и чанкинга пройдены.')

Проверки распознавания авторов и чанкинга пройдены.
